# Predicting Electric Vehicle Purchases: Modeling

Kaggle Playground Series S6E9. One notebook, versioned: v1 baselines →
v2 E01 budget-matched tuning (champion `e01_cat_2000x05` promoted via the
paired gate) → v3 E02 interactions → v5 E03 seed averaging →
**v7 E06 value identities**. Every run — kept
or rejected — has a row in `docs/4_experiment_ledger.md`; gates are frozen
there *before* execution, provably (the predeclaration commits precede the
results commits).

**Structure.** §1–3 set up config, data and the shared CV harness. §4
collects every completed experiment behind flags that default off — their
results are recorded in `docs/4_experiment_ledger.md`, and flipping a flag
reproduces one. §5–6 are the live work: the champion baseline and the
current experiment (E06). §7–10 gate, summarize, select, and submit. A run's
cost is therefore proportional to the *new* work, not the whole history.

EDA context (`docs/2_eda_insights.md`): top-heavy signal, one big
interaction (the subsidy gate), monotone ordinals, no missing values, no
drift — and CV↔LB confirmed by submission 1 (OOF 0.94177 → public
0.94169).

## 1. Config

In [ ]:
import json
import platform
import resource
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import lightgbm as lgb
from catboost import CatBoostClassifier

SEED = 42            # fold seed -- F1 never varies
N_SPLITS = 5         # fold definition F1 -- docs/4_experiment_ledger.md
TARGET = "Will_Buy_EV"
POSITIVE_CLASS = "Yes"
NOTEBOOK_VERSION = "v7"
BASELINE_CHAMPION = "e03_cat_int_avg5seeds"  # ledger promotion (CPU)
STANDING_CHAMPION_OOF = 0.94223       # its recorded OOF; §9 submission floor
E04_GPU_BASE = "e04_gpu_base"          # in-run GPU gate baseline
E04_SEEDS = [42, 7, 2026]              # stage-2 averaging seeds
E04_GUARD_SECONDS = 900                # predeclared stage-2 guard
SOURCE_DATASET_SLUG = "ev-adoption-behavior-and-range-anxiety"
USE_GPU = False                        # E06: CPU (GPU is screening-only)
CHAMPION_EXTRA_SEEDS = [7, 2026]       # E02 seed-average members
E03_SEEDS_3 = [42, 7, 2026]            # E03 3-seed average members
E03_SEEDS_5 = [42, 7, 2026, 13, 99]    # E03 5-seed average members
N_BOOT = 1000        # paired stratified bootstrap draws (predeclared)

# Mode flags (master standard §4). Historical sections default off --
# results recorded in docs/4_experiment_ledger.md; flip on to reproduce.
# Since v4 the champion carries interaction features, so its re-fit
# lives in the E03 section (it doubles as a seed-average member).
RUN_V1_SANITY = False
RUN_V2_STRONG = False
RUN_ANX_CATEGORICAL_AB = False
RUN_E01_TUNING = False
RUN_CHAMPION = True
RUN_E02 = False
RUN_E03 = False
RUN_E04 = False
RUN_E05 = False
RUN_E06 = True
RUN_SUBMISSION = True

# Every experiment section supplies its own in-run gate baseline, so the
# generic champion re-fit (§5) must not also run. Derived rather than
# enumerated: since v4 BASELINE_CHAMPION names a seed *average*, and
# re-fitting that as a single model would silently mislabel a run and any
# submission built from it. Add new RUN_E* flags here, not to §5.
EXPERIMENT_ACTIVE = RUN_E02 or RUN_E03 or RUN_E04 or RUN_E05 or RUN_E06

NUMERIC_FEATURES = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]
BASE_CATEGORICALS = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
]
ANX = "Range_Anxiety_Level"
ANX_MAP = {"Low": 0, "Medium": 1, "High": 2}  # monotone -- EDA §4
ALL_FEATURES = NUMERIC_FEATURES + BASE_CATEGORICALS + [ANX]
# E06: numerics whose exact value identifies a source row (EDA §10).
VALUE_ID_FEATURES = ["Annual_Income_USD", "Daily_Commute_km"]
VALUE_ID_CATEGORICALS = BASE_CATEGORICALS + [
    f"{col}_id" for col in VALUE_ID_FEATURES
]

print("python", platform.python_version())
print({m.__name__: m.__version__ for m in (np, pd, sklearn, lgb)})

## 2. Data Loading & Feature Frames

`Range_Anxiety_Level` is ordinal-encoded (strictly monotone — EDA §4; the
categorical A/B tied, ledger). `interactions=True` adds the three frozen
E02 subsidy crosses — target-free, computed identically on train and
test.

In [ ]:
def _find_data_dir() -> Path:
    """Locate the competition files on Kaggle or locally.

    Kaggle has mounted competition data at both
    /kaggle/input/competitions/<slug> and /kaggle/input/<slug> depending on
    the worker, so per master standard §12 the mount tree is walked rather
    than assumed when the known layouts miss.
    """
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e9"),
        Path("/kaggle/input/playground-series-s6e9"),
        Path("../data"),
        Path("data"),
    ]
    for cand in candidates:
        if (cand / "train.csv").exists():
            return cand
    mount = Path("/kaggle/input")
    if mount.exists():
        hits = sorted(mount.rglob("train.csv"))
        if hits:
            return hits[0].parent
    raise FileNotFoundError("train.csv not found in any known location")


DATA_DIR = _find_data_dir()
print(f"DATA_DIR = {DATA_DIR.resolve()}")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert train.shape == (668_665, 15) and test.shape == (286_571, 14)
assert train[ALL_FEATURES].isna().sum().sum() == 0
assert test[ALL_FEATURES].isna().sum().sum() == 0

y = (train[TARGET] == POSITIVE_CLASS).astype(int)


def make_features(
    df: pd.DataFrame,
    anx_as_categorical: bool = False,
    interactions: bool = False,
    features_v2: bool = False,
    value_ids: bool = False,
) -> pd.DataFrame:
    """Model-ready feature frame.

    Args:
        df: Raw train or test frame.
        anx_as_categorical: Keep Range_Anxiety_Level categorical instead
            of the default ordinal int encoding.
        interactions: Add the three frozen E02 subsidy crosses
            (docs/4_experiment_ledger.md, E02).
        features_v2: Additionally add the frozen E04 feature-v2 block
            (anxiety/income crosses and conditional charging).
        value_ids: Add string copies of VALUE_ID_FEATURES so CatBoost
            target-encodes the *exact* value (E06). The numeric
            columns are kept; only the identity is added.

    Returns:
        Feature frame with category dtypes on the base categoricals.
    """
    frame = df[ALL_FEATURES].copy()
    if anx_as_categorical:
        frame[ANX] = frame[ANX].astype("category")
    else:
        frame[ANX] = frame[ANX].map(ANX_MAP).astype("int8")
    if interactions:
        sub = (df["Subsidy_Available"] == "Yes").astype("int8")
        home = (df["Home_Charging_Possible"] == "Yes").astype("int8")
        frame["Subsidy_x_EnvConcern"] = (
            sub * df["Environmental_Concern_Level"]
        )
        frame["Subsidy_x_Income"] = sub * df["Annual_Income_USD"]
        frame["Subsidy_x_HomeCharging"] = sub * home
    if features_v2:
        anx_ord = df[ANX].map(ANX_MAP).astype("int8")
        stations = (
            df["Charging_Stations_Near_Home"]
            + df["Charging_Stations_Near_Work"]
        )
        frame["Anxiety_x_Subsidy"] = (2 - anx_ord) * sub
        frame["EnvConcern_x_Income"] = (
            df["Environmental_Concern_Level"] * df["Annual_Income_USD"]
        )
        frame["Total_Stations"] = stations
        frame["Stations_x_NoHomeCharging"] = stations * (1 - home)
    if value_ids:
        for col in VALUE_ID_FEATURES:
            frame[f"{col}_id"] = df[col].astype(str)
    for col in BASE_CATEGORICALS:
        frame[col] = frame[col].astype("category")
    return frame


X = make_features(train)
X_test = make_features(test)
X_int = make_features(train, interactions=True)
X_test_int = make_features(test, interactions=True)
X_v2 = make_features(train, interactions=True, features_v2=True)
X_test_v2 = make_features(test, interactions=True, features_v2=True)
X_v3 = make_features(train, interactions=True, value_ids=True)
X_test_v3 = make_features(test, interactions=True, value_ids=True)
print(X_v3.dtypes.to_string())
for col in VALUE_ID_FEATURES:
    seen = set(X_v3[f"{col}_id"])
    unseen = (~X_test_v3[f"{col}_id"].isin(seen)).mean()
    print(f"{col}: {len(seen)} train values; "
          f"{unseen:.4f} of test rows carry an unseen value")

def load_source_frame():
    """Load the raw CC0 source dataset if attached, else None.

    Provenance and usage rules: docs/7_source_dataset_provenance.md.
    """
    roots = [Path("/kaggle/input"), Path("../data"), Path("data")]
    hits = []
    for root in roots:
        if root.exists():
            hits += sorted(root.rglob("EV_Adoption_and_Range_Anxiety*.csv"))
    if not hits:
        print(
            "source dataset not attached to this kernel -- attach "
            f"{SOURCE_DATASET_SLUG} in kernel-metadata.json."
        )
        return None
    return pd.read_csv(hits[0])


def load_source_rows(src):
    """Source rows as extra training rows (E05).

    Rows with missing values are dropped so the training distribution
    keeps the competition's no-missing regime.

    Returns:
        (X_extra, y_extra) aligned to the interaction feature frame.
    """
    before = len(src)
    src = src.dropna(subset=ALL_FEATURES).reset_index(drop=True)
    assert set(ALL_FEATURES).issubset(src.columns)
    X_extra = make_features(src, interactions=True)
    y_extra = (src[TARGET] == POSITIVE_CLASS).astype(int)
    print(
        f"source rows: {before} loaded, {len(src)} usable after dropping "
        f"missing ({before - len(src)} dropped); positive rate "
        f"{y_extra.mean():.4f}"
    )
    return X_extra, y_extra


def build_source_lookup(src):
    """Per-income label statistics of the *source* rows (E06).

    Uses the source dataset's own labels only -- never the competition
    target -- so it needs no fold handling.

    Returns:
        DataFrame indexed by Annual_Income_USD with Src_Income_Rate and
        Src_Income_N.
    """
    s = src.dropna(subset=["Annual_Income_USD", TARGET])
    grp = s.groupby("Annual_Income_USD")[TARGET]
    return pd.DataFrame({
        "Src_Income_Rate": grp.apply(
            lambda v: float((v == POSITIVE_CLASS).mean())
        ),
        "Src_Income_N": grp.size(),
    })


def add_source_lookup(frame, df, lookup):
    """Append the source-income lookup to a feature frame (E06)."""
    out = frame.copy()
    key = df["Annual_Income_USD"]
    out["Src_Income_Rate"] = (
        key.map(lookup["Src_Income_Rate"]).fillna(-1.0).astype("float32")
    )
    out["Src_Income_N"] = (
        key.map(lookup["Src_Income_N"]).fillna(0).astype("int16")
    )
    return out


SOURCE_FRAME = load_source_frame() if (RUN_E05 or RUN_E06) else None
X_source, y_source = (
    load_source_rows(SOURCE_FRAME)
    if RUN_E05 and SOURCE_FRAME is not None
    else (None, None)
)
if RUN_E06 and SOURCE_FRAME is not None:
    SOURCE_LOOKUP = build_source_lookup(SOURCE_FRAME)
    X_v3s = add_source_lookup(X_v3, train, SOURCE_LOOKUP)
    X_test_v3s = add_source_lookup(X_test_v3, test, SOURCE_LOOKUP)
    print(
        f"source lookup: {len(SOURCE_LOOKUP)} incomes; train rows with a "
        f"match {(X_v3s['Src_Income_N'] > 0).mean():.4f}, test "
        f"{(X_test_v3s['Src_Income_N'] > 0).mean():.4f}"
    )


## 3. CV Harness (F1), Model Factories, Averaging

One harness for every model so OOF predictions align row-for-row. The
champion factory is parameterized by model seed for the E02 seed-average;
the fold split always uses `SEED` — F1 never varies.

In [ ]:
results = []
oof_store = {}
test_store = {}


def peak_rss_gb() -> float:
    """Process peak RSS in GB (ru_maxrss is bytes on macOS, KB on Linux)."""
    raw = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return raw / (1024**3 if sys.platform == "darwin" else 1024**2)


def _fold_iter(X_tr):
    skf = StratifiedKFold(
        n_splits=N_SPLITS, shuffle=True, random_state=SEED
    )
    return skf.split(X_tr, y)


def _register(name, oof, test_pred, wall_s):
    # A run name is a primary key: `results` is appended to while every
    # lookup is `next(r for r in results if r["run"] == name)`, which
    # returns the FIRST match, and oof_store[name] is overwritten. A
    # duplicate therefore silently splits one name across two different
    # vectors. Fail loudly instead.
    assert name not in oof_store, (
        f"duplicate run name {name!r} — run names must be unique within "
        "a run; see §1 on using literal names in historical sections"
    )
    fold_aucs = [
        roc_auc_score(y.iloc[va], oof[va]) for _, va in _fold_iter(X)
    ]
    row = {
        "run": name,
        "oof_auc": float(roc_auc_score(y, oof)),
        "fold_std": float(np.std(fold_aucs)),
        "fold_aucs": [round(float(a), 5) for a in fold_aucs],
        "wall_s": round(wall_s, 1),
        "peak_rss_gb": round(peak_rss_gb(), 2),
    }
    results.append(row)
    oof_store[name] = oof
    test_store[name] = test_pred
    print(
        f"{name:32s} OOF AUC {row['oof_auc']:.5f} ± {row['fold_std']:.5f} "
        f"| {row['wall_s']:7.1f}s | peak RSS {row['peak_rss_gb']:.2f} GB"
    )
    print(f"{'':32s} folds: {row['fold_aucs']}")
    return row


def run_cv(
    name, model_factory, X_tr, X_te, cat_features=None, extra_train=None
):
    """5-fold OOF CV on F1; stores aligned OOF and fold-mean test preds.

    Args:
        extra_train: Optional (X_extra, y_extra) appended to the
            *training* portion of every fold only. Validation folds
            stay pure competition data, so OOF remains measured on the
            competition distribution and comparable to every other run
            (master standard §5).
    """
    oof = np.zeros(len(X_tr))
    test_pred = np.zeros(len(X_te))
    t0 = time.time()
    for tr_idx, va_idx in _fold_iter(X_tr):
        model = model_factory()
        X_fit, y_fit = X_tr.iloc[tr_idx], y.iloc[tr_idx]
        if extra_train is not None:
            X_extra, y_extra = extra_train
            X_fit = pd.concat([X_fit, X_extra], ignore_index=True)
            y_fit = pd.concat([y_fit, y_extra], ignore_index=True)
            for col in BASE_CATEGORICALS:
                X_fit[col] = X_fit[col].astype("category")
        if cat_features is None:
            model.fit(X_fit, y_fit)
        else:
            model.fit(X_fit, y_fit, cat_features=cat_features)
        oof[va_idx] = model.predict_proba(X_tr.iloc[va_idx])[:, 1]
        test_pred += model.predict_proba(X_te)[:, 1] / N_SPLITS
    _register(name, oof, test_pred, time.time() - t0)
    return oof, test_pred


def register_average(name, members):
    """Register the element-wise mean of member OOF/test vectors."""
    oof = np.mean([oof_store[m] for m in members], axis=0)
    test_pred = np.mean([test_store[m] for m in members], axis=0)
    wall = sum(
        r["wall_s"] for r in results if r["run"] in members
    )
    return _register(name, oof, test_pred, wall)


def champion_factory(model_seed: int = SEED, **overrides):
    """The champion CatBoost config (2000 x 0.05), optionally on GPU.

    Args:
        model_seed: Model seed; the fold split always uses SEED.
        **overrides: Extra CatBoostClassifier parameters (E04 varies
            regularization here).

    Returns:
        An unfitted CatBoostClassifier.
    """
    params = dict(
        iterations=2000, learning_rate=0.05, random_seed=model_seed,
        verbose=0, allow_writing_files=False,
    )
    if USE_GPU:
        params.update(task_type="GPU", devices="0")
    params.update(overrides)
    return CatBoostClassifier(**params)

## 4. Historical Experiments *(all flags off — results live in the ledger)*

Kept for reproducibility and interpretation, not re-run by default: each
subsection's numbers are recorded in `docs/4_experiment_ledger.md`, and
flipping its flag in §1 reproduces it. Skim these for *what was already
settled*; the live experiment is §6.

### 4.1 v1 — Sanity Baselines

Constant floor 0.500; logistic 0.93809 ± 0.00081 (solver overflow warnings,
non-blocking); HGB default 0.94102 ± 0.00087 — the measured first fit
(scale override satisfied).

In [ ]:
if RUN_V1_SANITY:
    const_auc = roc_auc_score(y, np.full(len(y), float(y.mean())))
    print(f"v1a_constant: AUC {const_auc:.3f} (rankless floor)")

    def logistic_factory():
        pre = ColumnTransformer([
            ("num", StandardScaler(), NUMERIC_FEATURES + [ANX]),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore"),
                BASE_CATEGORICALS,
            ),
        ])
        return Pipeline([
            ("pre", pre),
            ("clf", LogisticRegression(max_iter=2000)),
        ])

    run_cv("v1b_logistic", logistic_factory, X, X_test)
    run_cv(
        "v1c_hgb_default",
        lambda: HistGradientBoostingClassifier(
            random_state=SEED, categorical_features="from_dtype"
        ),
        X,
        X_test,
    )

### 4.2 v2 — Strong Models and the ANX A/B

LightGBM default 0.94115 ± 0.00082; CatBoost default 0.94157 ± 0.00072
(working champion until E01). ANX ordinal-vs-categorical A/B: Δ +0.00008 —
tie, ordinal retained.

In [ ]:
if RUN_V2_STRONG:
    run_cv(
        "v2a_lightgbm_default",
        lambda: lgb.LGBMClassifier(random_state=SEED, verbose=-1),
        X,
        X_test,
    )
    run_cv(
        "v2b_catboost_default",
        lambda: CatBoostClassifier(
            random_seed=SEED, verbose=0, allow_writing_files=False
        ),
        X,
        X_test,
        cat_features=BASE_CATEGORICALS,
    )

if RUN_ANX_CATEGORICAL_AB:
    X_anxcat = make_features(train, anx_as_categorical=True)
    X_test_anxcat = make_features(test, anx_as_categorical=True)
    run_cv(
        "v2c_lightgbm_anx_categorical",
        lambda: lgb.LGBMClassifier(random_state=SEED, verbose=-1),
        X_anxcat,
        X_test_anxcat,
    )

### 4.3 E01 — Budget-Matched Configs

Seven frozen configs (ledger). Outcome: `e01_cat_2000x05` **0.94176**
promoted 5/5 folds, CI (+0.000145, +0.000239); every capacity increase
scored worse than its smaller sibling; Optuna therefore skipped; blend
closed by the 0.995 diversity bar (champion correlations 0.9961–0.9964).

In [ ]:
E01_CONFIGS = [
    (
        "e01_hgb_1000x05",
        lambda: HistGradientBoostingClassifier(
            max_iter=1000, learning_rate=0.05, early_stopping=False,
            random_state=SEED, categorical_features="from_dtype",
        ),
    ),
    (
        "e01_hgb_2000x03_63l",
        lambda: HistGradientBoostingClassifier(
            max_iter=2000, learning_rate=0.03, max_leaf_nodes=63,
            early_stopping=False, random_state=SEED,
            categorical_features="from_dtype",
        ),
    ),
    (
        "e01_lgbm_1000x05",
        lambda: lgb.LGBMClassifier(
            n_estimators=1000, learning_rate=0.05,
            random_state=SEED, verbose=-1,
        ),
    ),
    (
        "e01_lgbm_2000x03_63l",
        lambda: lgb.LGBMClassifier(
            n_estimators=2000, learning_rate=0.03, num_leaves=63,
            min_child_samples=50, random_state=SEED, verbose=-1,
        ),
    ),
    (
        "e01_lgbm_1000x05_127l",
        lambda: lgb.LGBMClassifier(
            n_estimators=1000, learning_rate=0.05, num_leaves=127,
            min_child_samples=100, colsample_bytree=0.8,
            random_state=SEED, verbose=-1,
        ),
    ),
    (
        "e01_cat_1000x10_d8",
        lambda: CatBoostClassifier(
            iterations=1000, learning_rate=0.1, depth=8,
            random_seed=SEED, verbose=0, allow_writing_files=False,
        ),
    ),
]

if RUN_E01_TUNING:
    for name, factory in E01_CONFIGS:
        cats = BASE_CATEGORICALS if "_cat_" in name else None
        run_cv(name, factory, X, X_test, cat_features=cats)

### 4.4 E02 — Champion-Improvement Candidates

Outcome (ledger): `e02_cat_interactions` **0.94204** promoted (5/5 folds,
CI +0.000209…+0.000317) and became champion; seed-averaging promoted but
superseded (0.94193); LightGBM interactions rejected by the gate despite
replicating the effect (0.94155 → 0.94182); 3000×0.035 tied — budget
direction closed.

In [ ]:
E02_CANDIDATES = [
    "e02_cat_interactions",
    "e02_cat_avg3seeds",
    "e02_cat_3000x035",
    "e02_lgbm_1000x05_interactions",
]

E02_SEED42_MEMBER = "e01_cat_2000x05"  # non-interaction, as E02 ran

if RUN_E02:
    # The seed-average members are non-interaction fits on X, so the
    # seed-42 member is the E01 champion, fit here under its literal
    # name. Using BASELINE_CHAMPION would KeyError (it names a run this
    # section never fits) or silently average incompatible features.
    run_cv(
        E02_SEED42_MEMBER,
        champion_factory,
        X,
        X_test,
        cat_features=BASE_CATEGORICALS,
    )
    run_cv(
        "e02_cat_interactions",
        champion_factory,
        X_int,
        X_test_int,
        cat_features=BASE_CATEGORICALS,
    )
    for extra_seed in CHAMPION_EXTRA_SEEDS:
        run_cv(
            f"e02_cat_s{extra_seed}",
            lambda s=extra_seed: champion_factory(s),
            X,
            X_test,
            cat_features=BASE_CATEGORICALS,
        )
    register_average(
        "e02_cat_avg3seeds",
        [E02_SEED42_MEMBER]
        + [f"e02_cat_s{s}" for s in CHAMPION_EXTRA_SEEDS],
    )
    run_cv(
        "e02_cat_3000x035",
        lambda: CatBoostClassifier(
            iterations=3000, learning_rate=0.035, random_seed=SEED,
            verbose=0, allow_writing_files=False,
        ),
        X,
        X_test,
        cat_features=BASE_CATEGORICALS,
    )
    run_cv(
        "e02_lgbm_1000x05_interactions",
        lambda: lgb.LGBMClassifier(
            n_estimators=1000, learning_rate=0.05,
            random_state=SEED, verbose=-1,
        ),
        X_int,
        X_test_int,
    )

**Insight:** the interaction hypothesis delivered the largest gain since
the defaults: `e02_cat_interactions` **0.94204** (+0.00027 over the in-run
champion re-fit), and the same crosses lifted LightGBM from its E01 best
0.94155 to 0.94182 — replication across families says it is the features,
not seed luck. Seed-averaging was worth a real but smaller +0.00016
(0.94193). The budget direction is exhausted: 3000×0.035 tied the champion
to the 5th decimal. Wall-clocks are clean Kaggle numbers this time.

### 4.5 E03 — Additivity Test

Outcome (ledger): interactions + seed-averaging proved additive — the
predeclared prediction of OOF 0.94220 was met **exactly** by the 3-seed
average; `e03_cat_int_avg5seeds` (0.94223) became champion, and averaging
plateaued past three seeds. Measured single-seed noise floor: 0.00013.

In [ ]:
E03_CANDIDATES = ["e03_cat_int_avg3seeds", "e03_cat_int_avg5seeds"]

E03_SEED42_MEMBER = "e03_cat_int_s42"

if RUN_E03:
    # Literal names only: BASELINE_CHAMPION is a moving pointer that now
    # names this section's own 5-seed average, so using it here would
    # write one name twice (single-seed row + average row). The seed-42
    # member is the same configuration E02 recorded as
    # `e02_cat_interactions`; it is renamed, not re-specified.
    run_cv(
        E03_SEED42_MEMBER,
        champion_factory,
        X_int,
        X_test_int,
        cat_features=BASE_CATEGORICALS,
    )
    for extra_seed in [s for s in E03_SEEDS_5 if s != SEED]:
        run_cv(
            f"e03_cat_int_s{extra_seed}",
            lambda s=extra_seed: champion_factory(s),
            X_int,
            X_test_int,
            cat_features=BASE_CATEGORICALS,
        )

    def _members(seeds):
        return [
            E03_SEED42_MEMBER if s == SEED else f"e03_cat_int_s{s}"
            for s in seeds
        ]

    register_average("e03_cat_int_avg3seeds", _members(E03_SEEDS_3))
    register_average("e03_cat_int_avg5seeds", _members(E03_SEEDS_5))

**Insight:** the additivity prediction was **exact**. The ledger predicted
OOF ≈ 0.94220 for interactions + seed-averaging; `e03_cat_int_avg3seeds`
delivered **0.94220**. The two effects act on genuinely different error
sources — features and seed variance — and compose without overlap.
Averaging then plateaus: five seeds add only +0.00003 (0.94223) for 65%
more compute. Individual seeds of the *same* config span 0.94196–0.94209
(0.00013), which is why averaging pays at all and why single-seed
differences below that spread mean nothing.

### 4.6 E04 — GPU Calibration, Regularization, Features v2 *(flag off since v6)*

Outcome (ledger): **null on both hypotheses**, and two findings that
outlived them — GPU is 8.9× faster but 0.00070 AUC *worse* on an
identical config (screening only, never for a champion), and GPU is not
bit-reproducible run-to-run (P(Δ>0) swung 0.648 → 0.879 on identical
inputs). Regularization moved OOF below the noise floor, falsifying the
earlier "regularization-side plateau" guess in §4.4.

### 4.6.1 Stage 1 — Single-Seed Candidates

Four candidates against `e04_gpu_base`, one variable each: two
`l2_leaf_reg` levels, one `random_strength`/`bagging_temperature`
combination, and the feature-v2 set.

In [ ]:
E04_STAGE1 = [
    ("e04_gpu_l2_10", {"l2_leaf_reg": 10}, "int"),
    ("e04_gpu_l2_30", {"l2_leaf_reg": 30}, "int"),
    (
        "e04_gpu_rs2_bag2",
        {"random_strength": 2.0, "bagging_temperature": 2.0},
        "int",
    ),
    ("e04_gpu_featv2", {}, "v2"),
]
FRAMES = {"int": (X_int, X_test_int), "v2": (X_v2, X_test_v2)}

if RUN_E04:
    # Calibration + gate baseline: champion config, GPU, seed 42.
    run_cv(
        E04_GPU_BASE,
        champion_factory,
        X_int,
        X_test_int,
        cat_features=BASE_CATEGORICALS,
    )
    base_row = next(r for r in results if r["run"] == E04_GPU_BASE)
    gpu_fit_seconds = base_row["wall_s"]
    print(
        f"GPU calibration: {gpu_fit_seconds:.0f}s per 5-fold fit "
        f"(CPU reference for this config: 3018s, OOF 0.94204)"
    )

    for name, overrides, frame_key in E04_STAGE1:
        X_tr, X_te = FRAMES[frame_key]
        run_cv(
            name,
            lambda o=overrides: champion_factory(**o),
            X_tr,
            X_te,
            cat_features=BASE_CATEGORICALS,
        )

**Insight:** **null on both hypotheses, and two facts worth more.**

*GPU is 8.9× faster and measurably worse.* Identical config, same folds:
GPU **0.94134** vs. CPU **0.94204** — a −0.00070 deficit at 341 s vs.
3018 s. That is 5.4× the single-seed noise floor and larger than every
gain this project has won combined (≈ +0.00046). GPU is therefore a
**screening tool only** here: explore cheaply on GPU, re-fit on CPU
before anything becomes a champion. The predeclared cross-device rule is
what stopped this run from submitting a worse model.

*Regularization is not the answer either.* `l2_leaf_reg` 10/30 moved OOF
by +0.00001/+0.00002 — below noise — and `random_strength`/
`bagging_temperature` hurt by 0.00030. This **falsifies** the earlier
speculation (§4.4 of this notebook) that "the plateau is
regularization-side, not capacity-starved": neither axis has headroom.

*Features v2 add nothing:* +0.00003, gate-rejected. The EDA §6
conditional-charging idea does not pay; E02's three crosses appear to
have captured the available feature signal.

### 4.6.2 Stage 2 — Averaging the Stage-1 Winner

Selection rule, predeclared: the highest-OOF stage-1 candidate that beats
`e04_gpu_base` is averaged over seeds {42, 7, 2026}; the baseline is
averaged over the same seeds so the gate compares like with like. Skipped
if nothing beat the baseline, or if the calibration guard tripped.

In [ ]:
E04_CANDIDATES = []

if RUN_E04:
    base_auc = next(
        r["oof_auc"] for r in results if r["run"] == E04_GPU_BASE
    )
    stage1_winners = sorted(
        (
            r for r in results
            if r["run"].startswith("e04_gpu_")
            and r["run"] != E04_GPU_BASE
            and r["oof_auc"] > base_auc
        ),
        key=lambda r: r["oof_auc"],
        reverse=True,
    )
    if not stage1_winners:
        print("stage 1: no candidate beat the GPU baseline — E04 records "
              "a null result and stage 2 is skipped.")
    elif gpu_fit_seconds > E04_GUARD_SECONDS:
        print(f"stage 2 skipped: calibration {gpu_fit_seconds:.0f}s "
              f"exceeds the predeclared {E04_GUARD_SECONDS}s guard.")
    else:
        winner = stage1_winners[0]["run"]
        print(f"stage 1 winner: {winner} — averaging over {E04_SEEDS}")
        spec = next(c for c in E04_STAGE1 if c[0] == winner)
        _, overrides, frame_key = spec
        X_tr, X_te = FRAMES[frame_key]
        for extra in [s for s in E04_SEEDS if s != SEED]:
            run_cv(
                f"{winner}_s{extra}",
                lambda s=extra, o=overrides: champion_factory(s, **o),
                X_tr,
                X_te,
                cat_features=BASE_CATEGORICALS,
            )
            run_cv(
                f"{E04_GPU_BASE}_s{extra}",
                lambda s=extra: champion_factory(s),
                X_int,
                X_test_int,
                cat_features=BASE_CATEGORICALS,
            )
        register_average(
            "e04_gpu_base_avg3",
            [E04_GPU_BASE]
            + [f"{E04_GPU_BASE}_s{s}" for s in E04_SEEDS if s != SEED],
        )
        register_average(
            "e04_gpu_best_avg3",
            [winner]
            + [f"{winner}_s{s}" for s in E04_SEEDS if s != SEED],
        )
        E04_CANDIDATES = ["e04_gpu_best_avg3"]

### 4.7 E05 — Does the Source Dataset Help?

Outcome (ledger): **null**, exactly as predeclared — `e05_cpu_plus_source`
0.94205 vs `e05_cpu_base` 0.94204, 95% CI (−0.000049, +0.000073),
P(Δ>0) 0.627. Frozen before execution **with the prior stated plainly:
this was expected to do nothing.** The source dataset
(`docs/7_source_dataset_provenance.md`) is CC0 and confirmed, but it is
itself synthetic and adds 9,466 usable rows to 668,665 — **1.4%**.

Fold-safe by construction: source rows enter the **training portion of
each fold only**, so OOF stays measured purely on competition data and
remains comparable to every earlier run. This is a screen — the champion
is a 5-seed average, so clearing this gate justifies an averaged
follow-up, not a submission.

In [ ]:
E05_CANDIDATES = []

if RUN_E05 and X_source is not None:
    run_cv(
        "e05_cpu_base",
        champion_factory,
        X_int,
        X_test_int,
        cat_features=BASE_CATEGORICALS,
    )
    run_cv(
        "e05_cpu_plus_source",
        champion_factory,
        X_int,
        X_test_int,
        cat_features=BASE_CATEGORICALS,
        extra_train=(X_source, y_source),
    )
    E05_CANDIDATES = ["e05_cpu_plus_source"]
elif RUN_E05:
    print("E05 skipped: source dataset not attached (see §2).")


**Insight:** the prediction held — Δ = +0.00001, inside the 0.00013
noise floor, 3/5 folds, CI spanning zero. Adding 1.4% synthetic rows from
a synthetic source teaches the model nothing 668k rows had not. Two
facts worth more than the null: `e05_cpu_base` reproduced kernel v4's
`e02_cat_interactions` at correlation **1.0000** (CPU fits are
bit-reproducible across kernel versions — the property GPU lacks), and
the two runs correlate at 0.9977, so there was no blend to consider
either. The source dataset stays as provenance only.


## 5. Champion Re-fit *(runs only when no experiment supplies its own baseline)*

`e02_cat_interactions` re-fit in-run so every gate comparison is
within-run. When E03 runs, its seed-42 member **is** this re-fit — fit
once, used for both.

In [ ]:
if RUN_CHAMPION and not EXPERIMENT_ACTIVE:
    # Only valid when BASELINE_CHAMPION names a single-model config; see
    # EXPERIMENT_ACTIVE in §1 for why this is derived, not enumerated.
    run_cv(
        BASELINE_CHAMPION,
        champion_factory,
        X_int,
        X_test_int,
        cat_features=BASE_CATEGORICALS,
    )

## 6. E06 — Value Identity: Exact Numeric Values as Categoricals

Frozen in `docs/4_experiment_ledger.md` before execution. The premise is
in EDA §10: `Annual_Income_USD` takes 13,214 distinct values, 97.9% of
which are values from the source dataset's 8,915 incomes, and the
*exact* value carries label information its magnitude does not — an
out-of-fold encoding of the value scores AUC 0.7072 alone vs 0.6812 for
100 quantile bins. Tree quantization (254 borders) cannot isolate one
value in 13k, so no model so far has used this.

Features v3 add string copies of income and commute as CatBoost
categoricals. CatBoost's ordered target statistics are computed inside
each training fold, so the encoding is leakage-safe by construction, and
values absent from a training fold get the prior — exactly what unseen
test values (0.58% of test incomes) will get. One arm adds a lookup of
the *source* rows' label rate per income, which uses no competition
target and needs no fold handling.


In [ ]:
E06_CANDIDATES = []

if RUN_E06:
    run_cv(
        "e06_cpu_base",
        champion_factory,
        X_int,
        X_test_int,
        cat_features=BASE_CATEGORICALS,
    )
    run_cv(
        "e06_cat_value_ids",
        champion_factory,
        X_v3,
        X_test_v3,
        cat_features=VALUE_ID_CATEGORICALS,
    )
    E06_CANDIDATES = ["e06_cat_value_ids"]
    if SOURCE_FRAME is not None:
        run_cv(
            "e06_cat_value_ids_src",
            champion_factory,
            X_v3s,
            X_test_v3s,
            cat_features=VALUE_ID_CATEGORICALS,
        )
        E06_CANDIDATES.append("e06_cat_value_ids_src")
    else:
        print("e06_cat_value_ids_src skipped: source dataset not attached.")


**Insight:** <!--INSIGHT:e06--> *(to fill from the executed run)*


## 7. Paired Promotion Gate

The standing predeclared gate vs. the in-run champion re-fit: fold wins
≥ 3/5, paired stratified bootstrap (B=1000, seed 42) 95% CI entirely
positive, P(Δ>0) ≥ 0.95. Bootstrap only for candidates above the champion
point estimate.

In [ ]:
def paired_gate(cand: str, champ: str, n_boot: int = N_BOOT) -> dict:
    """Predeclared paired promotion gate on aligned F1 OOF predictions."""
    oof_c, oof_h = oof_store[cand], oof_store[champ]
    fold_deltas = []
    for _, va_idx in _fold_iter(X):
        y_va = y.iloc[va_idx]
        fold_deltas.append(
            roc_auc_score(y_va, oof_c[va_idx])
            - roc_auc_score(y_va, oof_h[va_idx])
        )
    wins = int(sum(d > 0 for d in fold_deltas))

    rng = np.random.RandomState(SEED)
    pos = np.where(y.values == 1)[0]
    neg = np.where(y.values == 0)[0]
    deltas = np.empty(n_boot)
    for b in range(n_boot):
        idx = np.r_[
            rng.choice(pos, len(pos), replace=True),
            rng.choice(neg, len(neg), replace=True),
        ]
        y_b = y.values[idx]
        deltas[b] = roc_auc_score(y_b, oof_c[idx]) - roc_auc_score(
            y_b, oof_h[idx]
        )
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    p_pos = float((deltas > 0).mean())
    return {
        "fold_deltas": [round(float(d), 6) for d in fold_deltas],
        "fold_wins": wins,
        "ci95": (round(float(lo), 6), round(float(hi), 6)),
        "p_delta_pos": p_pos,
        "promoted": bool(wins >= 3 and lo > 0 and p_pos >= 0.95),
    }


gate_results = {}
ACTIVE_CANDIDATES = (
    E06_CANDIDATES if RUN_E06
    else E05_CANDIDATES if RUN_E05
    else E04_CANDIDATES if RUN_E04
    else E03_CANDIDATES if RUN_E03
    else E02_CANDIDATES
)
GATE_BASELINE = (
    "e06_cpu_base" if RUN_E06
    else "e05_cpu_base" if RUN_E05
    else "e04_gpu_base_avg3" if RUN_E04
    else BASELINE_CHAMPION
)
if ACTIVE_CANDIDATES and GATE_BASELINE in oof_store:
    champ_auc = next(
        r["oof_auc"] for r in results if r["run"] == GATE_BASELINE
    )
    challengers = [
        r["run"]
        for r in results
        if r["run"] in ACTIVE_CANDIDATES and r["oof_auc"] > champ_auc
    ]
    print(
        f"gate baseline {GATE_BASELINE} OOF AUC {champ_auc:.5f}; "
        f"point-estimate challengers: {challengers or 'none'}"
    )
    for cand in challengers:
        gate_results[cand] = paired_gate(cand, GATE_BASELINE)
        g = gate_results[cand]
        print(
            f"{cand:32s} fold wins {g['fold_wins']}/5 | "
            f"95% CI {g['ci95']} | P(d>0) {g['p_delta_pos']:.3f} "
            f"| promoted: {g['promoted']}"
        )

**Insight:** `e04_gpu_best_avg3` won 4/5 folds but its 95% CI
**(−0.000034, +0.000051)** spans zero and P(Δ>0) = 0.648 — **not
promoted**, exactly the case the gate exists for. A fold-count-only rule
(4/5 looks convincing) would have promoted a candidate whose true effect
is indistinguishable from zero, and the run correctly wrote **no
submission**: the champion `e03_cat_int_avg5seeds` lives in an earlier
kernel version, so this run contributes evidence only.

## 8. Summary, Sanity Checks, Diversity

In [ ]:
summary = pd.DataFrame(results)
if not summary.empty:
    summary = (
        summary.sort_values("oof_auc", ascending=False)
        .reset_index(drop=True)
    )
summary


In [ ]:
def candidate_sanity_checks(name: str) -> dict:
    """Finite, bounded, non-degenerate predictions; quantile comparison."""
    oof, test_pred = oof_store[name], test_store[name]
    return {
        "finite": bool(
            np.isfinite(oof).all() and np.isfinite(test_pred).all()
        ),
        "in_range": bool(
            (oof >= 0).all()
            and (oof <= 1).all()
            and (test_pred >= 0).all()
            and (test_pred <= 1).all()
        ),
        "oof_unique": int(np.unique(oof).size),
        "test_unique": int(np.unique(test_pred).size),
        "oof_q05_50_95": np.quantile(oof, [0.05, 0.5, 0.95])
        .round(4)
        .tolist(),
        "test_q05_50_95": np.quantile(test_pred, [0.05, 0.5, 0.95])
        .round(4)
        .tolist(),
    }


for name in oof_store:
    print(name, candidate_sanity_checks(name))

oof_corr = pd.DataFrame(oof_store).corr().round(4)
print("\nOOF Pearson correlation (diversity bar: blend only if <= 0.995):")
print(oof_corr.to_string())

**Insight:** all runs pass sanity; the champion's test predictions are
fully distinct (286,571 unique values). Diversity vs. the champion:
`e03_cat_int_avg3seeds` 0.9999, `e02_cat_interactions` 0.9992 — as
expected for nested averages of one config, far above the 0.995 bar, so
blending remains closed. The seed spread (0.00013) is now the reference
scale: any future single-seed "improvement" smaller than that is noise
until it survives the paired gate.

## 9. Champion Selection & Prediction Artifacts

The champion stays `e01_cat_2000x05` unless a candidate cleared the gate;
seed components and the logistic floor are excluded by predeclaration.

In [ ]:
EXCLUDED_FROM_CANDIDACY = (
    {"v1b_logistic"}
    | {f"e02_cat_s{s}" for s in CHAMPION_EXTRA_SEEDS}
    | {f"e03_cat_int_s{s}" for s in E03_SEEDS_5}
)
promoted = [c for c, g in gate_results.items() if g["promoted"]]
# Whether THIS run may write a submission artifact. A printed warning
# is advice; only this flag actually stops §10 from writing the file.
SUBMIT_OK = True
if GATE_BASELINE == "e05_cpu_base":
    # Predeclared: E05 is a screen against a single-seed baseline,
    # while the champion is a 5-seed average -- so E05 never takes
    # the submission slot regardless of outcome.
    CHAMPION_NAME = BASELINE_CHAMPION
    SUBMIT_OK = False
    print(
        "E05 is a screen: champion unchanged "
        f"({CHAMPION_NAME}); promoted={promoted or 'none'}"
    )
elif promoted and RUN_E04:
    # Cross-device rule (docs/0_coding_standards.md): a GPU winner only
    # takes the submission slot if it also exceeds the standing CPU
    # champion's recorded OOF -- an observation, not a gate claim.
    STANDING_CPU_CHAMPION_OOF = STANDING_CHAMPION_OOF
    best = max(
        promoted,
        key=lambda c: next(
            r["oof_auc"] for r in results if r["run"] == c
        ),
    )
    best_auc = next(r["oof_auc"] for r in results if r["run"] == best)
    if best_auc > STANDING_CPU_CHAMPION_OOF:
        CHAMPION_NAME = best
        print(
            f"GPU winner {best} ({best_auc:.5f}) exceeds the standing CPU "
            f"champion ({STANDING_CPU_CHAMPION_OOF:.5f}) -- taking the "
            "submission slot (cross-device observation)."
        )
    else:
        CHAMPION_NAME = best
        SUBMIT_OK = False
        print(
            f"GPU winner {best} ({best_auc:.5f}) cleared its in-run gate "
            f"but does NOT exceed the standing CPU champion "
            f"({STANDING_CPU_CHAMPION_OOF:.5f}). Configuration evidence "
            "only -- no submission written."
        )
elif promoted:
    # Same-device rule: the in-run gate baseline is a single seed, but
    # the standing champion may be a seed average with a higher OOF. A
    # promoted candidate takes the submission slot only if it also beats
    # that recorded OOF; otherwise this run is evidence for an averaged
    # follow-up, not a submission.
    CHAMPION_NAME = max(
        promoted,
        key=lambda c: next(
            r["oof_auc"] for r in results if r["run"] == c
        ),
    )
    best_auc = next(
        r["oof_auc"] for r in results if r["run"] == CHAMPION_NAME
    )
    if best_auc > STANDING_CHAMPION_OOF:
        print(
            f"gate-promoted champion: {CHAMPION_NAME} ({best_auc:.5f} > "
            f"standing {STANDING_CHAMPION_OOF:.5f})"
        )
    else:
        SUBMIT_OK = False
        print(
            f"{CHAMPION_NAME} ({best_auc:.5f}) cleared its in-run gate but "
            f"does not exceed the standing champion's OOF "
            f"({STANDING_CHAMPION_OOF:.5f}, {BASELINE_CHAMPION}) -- "
            "evidence only, no submission written."
        )
else:
    CHAMPION_NAME = BASELINE_CHAMPION
    print(
        f"no candidate cleared the gate; champion stays {CHAMPION_NAME}"
    )
champ_row = next(
    (r for r in results if r["run"] == CHAMPION_NAME), None
)
if champ_row is None:
    # Fallback path: the standing champion was promoted in an earlier
    # kernel version and is not re-fit here, so this run holds no
    # predictions for it (the submission cell handles that).
    print(
        f"champion {CHAMPION_NAME} carries over from an earlier kernel "
        "version; no OOF row in this run."
    )
else:
    print(f"champion OOF AUC {champ_row['oof_auc']:.5f}")

# Local repo: ../predictions. On Kaggle: the kernel working dir, so
# `kaggle kernels output` can retrieve the matrices (docs/0 execution rule).
PRED_DIR = (
    Path("../predictions") if Path("../predictions").is_dir() else Path(".")
)
for name in oof_store:
    np.save(PRED_DIR / f"{name}_oof.npy", oof_store[name])
    np.save(PRED_DIR / f"{name}_test.npy", test_store[name])
print(f"aligned prediction matrices saved to {PRED_DIR.resolve()}")

## 10. Submission

In [ ]:
if RUN_SUBMISSION and not SUBMIT_OK:
    print(
        "no submission written: §9 ruled this run evidence-only "
        f"(champion {CHAMPION_NAME})."
    )
elif RUN_SUBMISSION and CHAMPION_NAME not in test_store:
    print(
        f"no submission written: champion {CHAMPION_NAME} was not fit "
        "in this run (its predictions live in an earlier kernel "
        "version). This run contributes evidence only."
    )
elif RUN_SUBMISSION:
    submission = pd.DataFrame(
        {"id": test["id"], TARGET: test_store[CHAMPION_NAME]}
    )
    assert submission.shape == sample_submission.shape
    assert (
        submission["id"].values == sample_submission["id"].values
    ).all()
    submission.to_csv("submission.csv", index=False)
    print(
        f"submission.csv written from {CHAMPION_NAME} "
        f"(notebook {NOTEBOOK_VERSION}); range "
        f"[{submission[TARGET].min():.4f}, "
        f"{submission[TARGET].max():.4f}]"
    )

## Reproducibility Snapshot

In [ ]:
snapshot = {
    "generated_utc": datetime.now(timezone.utc).isoformat(
        timespec="seconds"
    ),
    "notebook_version": NOTEBOOK_VERSION,
    "seed": SEED,
    "device": "GPU" if USE_GPU else "CPU",
    "fold_definition": (
        f"F1: StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, "
        f"random_state={SEED})"
    ),
    "flags": {
        "v1": RUN_V1_SANITY, "v2": RUN_V2_STRONG,
        "anx_ab": RUN_ANX_CATEGORICAL_AB, "e01": RUN_E01_TUNING,
        "champion": RUN_CHAMPION, "e02": RUN_E02, "e03": RUN_E03, "e04": RUN_E04, "e05": RUN_E05,
        "e06": RUN_E06,
    },
    "champion": CHAMPION_NAME,
    "gate_results": gate_results,
    "results": results,
    "versions": {
        m.__name__: m.__version__ for m in (np, pd, sklearn, lgb)
    },
    "python": platform.python_version(),
}
print(json.dumps(snapshot, indent=2))